
# OpenAI Agents SDK — Tools & Function Calling Practical Notebook

This notebook follows the practical concepts from the lesson on **Tools and Function Calling**.

We will build a math agent that can use custom Python functions as tools.

Topics covered:

- Hosted tools
- Custom function tools
- `@function_tool`
- Type hints and tool schemas
- Docstrings as tool descriptions
- Registering tools with an agent
- Multi-step tool calling
- The agentic tool-calling loop
- Agents as tools (conceptual overview)
- OpenAI Agents SDK vs. LangChain tool calling


## 1. Install Dependencies

In [ ]:
!pip install -U openai-agents


## 2. API Key Setup

The Agents SDK uses your OpenAI API key.

Set `OPENAI_API_KEY` in your environment before running the examples.

**Windows PowerShell**
```powershell
$env:OPENAI_API_KEY="your-api-key"
```

**macOS/Linux**
```bash
export OPENAI_API_KEY="your-api-key"
```

Do not commit your API key to GitHub or other source-control systems.


In [ ]:

import os

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set.")

print("API key detected.")


# 3. The Three Types of Tools


The OpenAI Agents SDK supports three broad tool categories.

### 1. Hosted tools

Tools provided by OpenAI, such as capabilities for web search or other hosted functionality.

### 2. Function tools

Your own Python functions exposed to the model as callable tools.

This is the main focus of this notebook.

### 3. Agents as tools

One agent can be exposed as a tool that another agent can call. This is useful for delegating work to specialized agents.

For this foundational notebook, we will focus primarily on **function tools**.


# 4. Create Your First Function Tool

In [ ]:

from agents import Agent, Runner, function_tool

@function_tool
def add(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b

print("Tool created:", add)



### What does `@function_tool` do?

The decorator converts a normal Python function into an Agents SDK function tool.

The SDK uses:

- The function name
- Type hints
- Function parameters
- The docstring

to generate information that the model can use when deciding whether and how to call the tool.

For example:

```python
@function_tool
def add(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b
```

The model can understand that:

- The tool is called `add`.
- It requires `a`.
- It requires `b`.
- Both values should be numbers.
- The tool adds the two values.


# 5. Create More Math Tools

In [ ]:

@function_tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers and return the result."""
    return a * b


@function_tool
def divide(a: float, b: float) -> float:
    """Divide a by b. Returns an error message if b is zero."""
    if b == 0:
        return "Error: cannot divide by zero."
    return a / b


@function_tool
def square_root(a: float) -> float:
    """Calculate the square root of a non-negative number."""
    if a < 0:
        return "Error: cannot calculate the square root of a negative number."

    return a ** 0.5


print("Math tools created successfully.")


# 6. Register Tools with an Agent

In [ ]:

math_agent = Agent(
    name="Math Agent",
    model="gpt-5.5",
    instructions=(
        "You are a helpful math assistant. "
        "Use the available tools whenever mathematical calculations are required. "
        "Explain the final answer clearly."
    ),
    tools=[
        add,
        multiply,
        divide,
        square_root,
    ],
)

print("Math agent created.")



The agent now has four capabilities:

```text
Math Agent
│
├── add
├── multiply
├── divide
└── square_root
```

The model can decide which tool is appropriate based on the user's question.


# 7. Run a Simple Tool-Calling Question

In [ ]:

result = Runner.run_sync(
    math_agent,
    "What is 15 multiplied by 8?"
)

print(result.final_output)



### What happened?

Conceptually, the flow is:

```text
User
  │
  │ "What is 15 multiplied by 8?"
  ▼
Agent
  │
  ▼
LLM
  │
  │ Decides that multiplication is required
  ▼
multiply(15, 8)
  │
  │ Returns 120
  ▼
LLM
  │
  ▼
Final natural-language answer
```

The SDK handles the function-calling plumbing.


# 8. Multi-Step Tool Calling

In [ ]:

result = Runner.run_sync(
    math_agent,
    "Calculate 15 multiplied by 8, then divide the result by 3."
)

print(result.final_output)



This question can require multiple tool calls.

A possible execution flow is:

```text
User Question
     ↓
LLM
     ↓
multiply(15, 8)
     ↓
120
     ↓
LLM reasons again
     ↓
divide(120, 3)
     ↓
40
     ↓
LLM
     ↓
Final Answer
```

This is an important characteristic of agents: the model can repeatedly reason, call tools, observe results, and continue until it can produce a final response.


# 9. Test All Math Operations

In [ ]:

questions = [
    "Add 25 and 17.",
    "Multiply 12 by 9.",
    "Divide 100 by 4.",
    "Calculate the square root of 144.",
    "Calculate 15 multiplied by 8 and then divide the result by 3.",
]

for question in questions:
    print("=" * 80)
    print("QUESTION:", question)

    result = Runner.run_sync(math_agent, question)

    print("ANSWER:", result.final_output)


# 10. Test an Edge Case

In [ ]:

result = Runner.run_sync(
    math_agent,
    "What is 100 divided by 0?"
)

print(result.final_output)



The tool itself handles the invalid operation:

```python
if b == 0:
    return "Error: cannot divide by zero."
```

This demonstrates an important principle:

> Put deterministic business logic and validation inside your tools whenever possible.

The LLM decides **when** to use the tool, while the Python function performs the actual operation.


# 11. Understanding the Tool Schema


The model needs to know how a tool can be called.

For this function:

```python
@function_tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers and return the result."""
    return a * b
```

the SDK can derive information conceptually similar to:

```json
{
  "name": "multiply",
  "description": "Multiply two numbers and return the result.",
  "parameters": {
    "a": "number",
    "b": "number"
  }
}
```

You normally do **not** have to manually write this schema when using `@function_tool`.

The type hints and docstring help the SDK describe the tool to the model.


# 12. The Agentic Tool-Calling Loop


The overall process looks like this:

```text
User Question
      ↓
Agent
      ↓
LLM
      ↓
Does the task require a tool?
      │
   ┌──┴──┐
   │     │
  No    Yes
   │     │
   │     ▼
   │   Tool Call
   │     │
   │     ▼
   │   Python Tool
   │     │
   │     ▼
   │   Tool Result
   │     │
   └─────┤
         ▼
       LLM
         │
         ▼
   More tools needed?
      │
   ┌──┴──┐
  Yes   No
   │     │
   └─loop │
         ▼
    Final Answer
```

The Agents SDK Runner manages this execution process.


# 13. Hosted Tools — Conceptual Example


Hosted tools are capabilities provided by OpenAI.

Examples can include capabilities such as:

- Web search
- File search
- Other hosted model tools

The main idea is that you do not implement the underlying capability yourself.

Instead of writing an entire web-search engine, for example, you use the supported hosted tool and allow the model to decide when it is useful.

The exact hosted-tool API and availability can change over time, so consult the current Agents SDK documentation when implementing production code.


# 14. Agents as Tools — Conceptual Example


An agent can also be exposed as a tool to another agent.

Conceptually:

```text
Manager Agent
     │
     ├── Research Agent
     ├── Math Agent
     └── Writing Agent
```

The manager can delegate specialized work to another agent.

For example:

```python
manager = Agent(
    name="Manager",
    instructions="Delegate specialized tasks to the appropriate agent.",
    tools=[
        specialist_agent.as_tool(...)
    ]
)
```

This pattern is useful for multi-agent architectures, but it is beyond the main scope of this foundational notebook.


# 15. LangChain vs. OpenAI Agents SDK — Tool Definition


### LangChain

A custom tool commonly uses:

```python
from langchain_core.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b
```

### OpenAI Agents SDK

The equivalent concept is:

```python
from agents import function_tool

@function_tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b
```

The decorators have different names, but the underlying idea is similar:

> Turn a Python function into a capability the model can call.


# 16. Practical Comparison


| Step | LangChain | OpenAI Agents SDK |
|---|---|---|
| Model setup | Model configured separately | Model configured in `Agent` |
| Tool decorator | `@tool` | `@function_tool` |
| Agent definition | Larger orchestration abstraction | Lightweight `Agent` abstraction |
| Execution | Agent executor / framework flow | `Runner.run_sync()` or `Runner.run()` |
| Final output | Often requires message/output handling | `result.final_output` |

The choice depends on your project and requirements.

For a focused OpenAI-based agent application, the Agents SDK can feel lightweight and straightforward. LangChain provides a broader ecosystem and more generalized abstractions.


# 17. Practical Exercise — Build a Temperature Tool

In [ ]:

# TODO:
# Create a function tool that converts Celsius to Fahrenheit.
#
# Formula:
# Fahrenheit = (Celsius * 9/5) + 32
#
# Requirements:
# 1. Use @function_tool.
# 2. Add a type hint.
# 3. Add a useful docstring.
# 4. Return the converted temperature.

# Your code here


# 18. Practical Exercise — Add the Tool to an Agent

In [ ]:

# TODO:
# 1. Create an agent with your temperature tool.
# 2. Ask:
#    "Convert 25 degrees Celsius to Fahrenheit."
# 3. Print result.final_output.

# Your code here


# 19. Practical Exercise — Multi-Step Calculation

In [ ]:

# TODO:
# Ask the math agent a question that requires at least two calculations.
#
# Example:
# "Multiply 25 by 4, then divide the result by 5."
#
# Run the agent and print the final answer.

# Your code here


# 20. Key Takeaways


1. **Tools give agents the ability to act.**
2. The Agents SDK supports hosted tools, function tools, and agents as tools.
3. `@function_tool` converts a Python function into a tool.
4. Type hints help define the expected parameters.
5. Docstrings help describe the tool to the model.
6. The agent receives the tools through the `tools` list.
7. The model decides when a tool is needed.
8. The Runner manages the tool-calling loop.
9. Tool calls can happen multiple times during one agent run.
10. The Python function performs deterministic work, while the LLM handles reasoning and tool selection.
11. Agents as tools enable delegation between specialized agents.
12. Compared with LangChain, the OpenAI Agents SDK provides a relatively lightweight agent abstraction for OpenAI-focused applications.
